# Construction Materials Cost Planner - Analysis Notebook
**Student:** [Your Name]

This notebook explains the analysis behind the application. Run all cells from the project folder (`Kernel > Restart & Run All`). It uses the same functions as `main.py`.

## 1. Module 1 - required material quantities
For each concrete element:
- net volume = length x width x height
- gross volume = net volume x (1 + waste % / 100)
- cement = 300 kg/m3, sand = 0.5 m3/m3, aggregate = 0.8 m3/m3 of gross volume

In [ ]:
import pandas as pd
from module1_calculator import ElementManager
from module2_analysis import *

manager = ElementManager()
manager.load_from_csv()
df_elements = pd.DataFrame(manager.list_elements())
df_elements

In [ ]:
# Totals per site
pd.DataFrame(manager.site_summary())

## 2. Module 2 - cleaning the data
Rules applied with pandas: prices must be > 0 and in the expected unit (cement kg, sand/aggregate m3); duplicate `transaction_id`s are dropped (first kept); missing or non-positive quantities, invalid dates, unknown element IDs and supplier/material pairs without a price are rejected. Every rejected record is listed with its reason.

In [ ]:
df_merged, df_prices, df_rejected = load_and_clean_data(valid_element_ids=set(manager.elements))
print(len(df_merged), "valid purchase rows;", len(df_rejected), "rejected")
df_rejected

## 3. Cost per site, supplier and month
cost = quantity x unit price, after merging purchases with prices on `supplier_id` + `material`, and with elements on `element_id` (to get the site).

In [ ]:
calculate_costs_summary(df_merged, df_elements)

## 4. Required vs purchased
Required quantities come from Module 1; purchased quantities are summed per element and material. Units are consistent (cement kg, sand/aggregate m3). Negative variance = **Shortage**, otherwise **Surplus**.

In [ ]:
comp = compare_required_vs_purchased(df_elements, df_merged)
display(comp)
comp["status"].value_counts()

## 5. NumPy price sensitivity
Required totals and the **lowest** price per material are stored as NumPy arrays; total cost is their dot product. One material's price is then raised by a chosen percentage.

In [ ]:
res = numpy_price_sensitivity_analysis(df_elements, df_prices, "Cement", 15)
res

In [ ]:
# Effect of a +15% rise for each material
import pandas as pd
pd.DataFrame([numpy_price_sensitivity_analysis(df_elements, df_prices, m, 15) for m in MATERIALS])[
    ["material", "percentage", "baseline_cost", "inflated_cost", "cost_increase"]]

## 6. Charts
Four charts are saved as PNG files in `outputs/`: monthly spend (line), spend per supplier (bar), quantity vs cost per material (scatter), and distribution of transaction costs (histogram).

In [ ]:
from IPython.display import Image, display
paths = generate_matplotlib_charts(df_merged, output_dir="outputs")
for p in paths:
    display(Image(p, width=520))

## 7. Tests (3 valid, 4 invalid)
`tests.py` records the expected and actual result of each case.

In [ ]:
%run tests.py

## 8. Conclusions
*(Write your own interpretation here - for example: which elements are in shortage and by how much, which supplier cost more, and which material's price rise matters most for the budget. Base it on the tables and charts above.)*